In [ ]:
# Just One time. but after install this library you have to restartt the kernel
#!pip install beautifulsoup4 serpapi requests dotenv google-search-results

In [ ]:
import os
import requests
import time
import yaml
from serpapi import GoogleSearch
from dotenv import load_dotenv
from IPython.display import display, HTML, clear_output
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# --- 1. Load Configuration ---

def load_config(config_path='config.yaml'):
    """Memuat parameter dari file YAML."""
    print(f"--- ⚙️ Memuat konfigurasi dari {config_path} ---")
    try:
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
        print("✅ Konfigurasi berhasil dimuat.")
        
        # Display configuration in a nice format for notebook
        display(HTML(f"""
        <div style="background-color: #1e1e1e; color: #ffffff; padding: 10px; border-radius: 5px; margin: 10px 0; border: 1px solid #333;">
            <h4 style="color: #4fc3f7;">📋 Konfigurasi yang dimuat:</h4>
            <ul style="color: #e0e0e0;">
                <li><strong style="color: #81c784;">Images per dish:</strong> {config.get('images_per_dish', 'Not set')}</li>
                <li><strong style="color: #81c784;">Save directory:</strong> {config.get('save_directory', 'Not set')}</li>
                <li><strong style="color: #81c784;">Target dishes:</strong> {len(config.get('target_queries', {}))} kategori</li>
            </ul>
        </div>
        """))
        
        return config
    except FileNotFoundError:
        print(f"❌ ERROR: File konfigurasi tidak ditemukan di {config_path}")
        return None
    except Exception as e:
        print(f"❌ ERROR: Gagal memuat file konfigurasi: {e}")
        return None

In [ ]:
# --- 2. Main Scraper Logic ---

def download_image(url, save_path, downloaded_urls):
    """Mengunduh satu gambar dan menyimpannya, menghindari duplikat."""
    if url in downloaded_urls:
        return False
        
    try:
        response = requests.get(url, timeout=15, stream=True)
        response.raise_for_status()
        
        with open(save_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        
        downloaded_urls.add(url)
        return True
    except requests.exceptions.RequestException:
        return False

def scrape_images_for_dish(dish_name, queries, num_images_total, save_dir, api_key):
    """
    Melakukan scraping gambar untuk satu hidangan menggunakan beberapa variasi query.
    """
    display(HTML(f"""
    <div style="background-color: #2d2d2d; color: #ffffff; padding: 15px; border-radius: 8px; margin: 10px 0; border: 1px solid #444;">
        <h3 style="color: #ff9800;">🚀 Memulai scrape untuk kategori: <strong style="color: #4fc3f7;">{dish_name}</strong></h3>
        <p style="color: #e0e0e0;">Target: <strong style="color: #81c784;">{num_images_total}</strong> gambar dengan <strong style="color: #81c784;">{len(queries)}</strong> variasi query</p>
    </div>
    """))
    
    dish_folder = os.path.join(save_dir, dish_name)
    os.makedirs(dish_folder, exist_ok=True)
    print(f"📁 Menyimpan gambar ke: {dish_folder}")

    downloaded_urls = set()
    total_downloaded_count = 0
    
    images_per_query = num_images_total // len(queries)

    for i, query in enumerate(queries, 1):
        if total_downloaded_count >= num_images_total:
            break
            
        print(f"\n🔍 Query {i}/{len(queries)}: '{query}'...")
        
        params = {
            "engine": "google_images",
            "q": query,
            "api_key": api_key,
            "num": 100
        }

        try:
            search = GoogleSearch(params)
            results = search.get_dict()
            image_results = results.get("images_results", [])
        except Exception as e:
            print(f"   ❌ Error saat mengambil hasil dari SerpApi untuk '{query}': {e}")
            continue

        if not image_results:
            print(f"   ⚠️ Tidak ada hasil gambar yang ditemukan untuk '{query}'.")
            continue
            
        query_download_count = 0
        for image_info in image_results:
            if total_downloaded_count >= num_images_total or query_download_count >= images_per_query:
                break

            img_url = image_info.get('original')
            if not img_url:
                continue

            # Clean the query string for filename
            clean_query = query.replace(' ', '_').replace('"', '').replace("'", "")
            file_name = f"{clean_query}_{query_download_count + 1}.jpg"
            file_path = os.path.join(dish_folder, file_name)
            
            if download_image(img_url, file_path, downloaded_urls):
                total_downloaded_count += 1
                query_download_count += 1
                
                # Update progress with better formatting for notebook
                progress_percent = (total_downloaded_count / num_images_total) * 100
                print(f"\r   ✅ Progress: {total_downloaded_count}/{num_images_total} ({progress_percent:.1f}%) gambar", end="", flush=True)
                time.sleep(0.1)
    
    # Final summary with nice formatting
    display(HTML(f"""
    <div style="background-color: #1a1a1a; color: #ffffff; padding: 10px; border-radius: 5px; margin: 10px 0; border: 1px solid #333;">
        <h4 style="color: #4caf50;">✨ Selesai scrape untuk <strong style="color: #4fc3f7;">{dish_name}</strong></h4>
        <p style="color: #e0e0e0;">📊 Total berhasil diunduh: <strong style="color: #ff9800;">{total_downloaded_count}</strong> gambar</p>
        <p style="color: #e0e0e0;">📁 Lokasi: <span style="color: #81c784;">{dish_folder}</span></p>
    </div>
    """))


In [ ]:
# --- 3. Setup and Configuration Display ---

def setup_environment():
    """Setup environment variables dan menampilkan status."""
    print("🔧 Setting up environment...")
    
    # Load environment variables
    load_dotenv('../.env')  # Adjust path as needed
    api_key = os.getenv('SERPAPI_API_KEY')
    
    if not api_key:
        display(HTML("""
        <div style="background-color: #d32f2f; color: #ffffff; padding: 15px; border-radius: 5px; border: 1px solid #b71c1c;">
            <h4 style="color: #ffcdd2;">❌ SERPAPI_API_KEY tidak ditemukan!</h4>
            <p style="color: #ffebee;">Pastikan untuk:</p>
            <ol style="color: #ffebee;">
                <li>Buat file <code style="background-color: #424242; color: #ffeb3b; padding: 2px 4px; border-radius: 3px;">.env</code> di parent directory</li>
                <li>Tambahkan: <code style="background-color: #424242; color: #4caf50; padding: 2px 4px; border-radius: 3px;">SERPAPI_API_KEY=your_api_key_here</code></li>
                <li>Atau set langsung di notebook: <code style="background-color: #424242; color: #4caf50; padding: 2px 4px; border-radius: 3px;">os.environ['SERPAPI_API_KEY'] = 'your_key'</code></li>
            </ol>
        </div>
        """))
        return None
    else:
        display(HTML("""
        <div style="background-color: #2e7d32; color: #ffffff; padding: 10px; border-radius: 5px; border: 1px solid #1b5e20;">
            <h4 style="color: #c8e6c9;">✅ SERPAPI_API_KEY berhasil dimuat!</h4>
        </div>
        """))
        return api_key

In [ ]:
# --- 4. Main Execution Function ---

def run_scraper(config_path='config.yaml'):
    """Fungsi utama untuk menjalankan seluruh proses scraping di notebook."""
    
    # Setup
    api_key = setup_environment()
    if not api_key:
        return
    
    # Load config
    config = load_config(config_path)
    if not config:
        return

    # Extract configuration
    save_directory = config.get('save_directory', '../data/raw/')
    os.makedirs(save_directory, exist_ok=True)
    
    target_queries = config.get('target_queries', {})
    images_per_dish = config.get('images_per_dish', 100)
    
    if not target_queries:
        print("❌ Tidak ada target queries yang ditemukan dalam konfigurasi!")
        return
    
    display(HTML(f"""
    <div style="background-color: #1a1a1a; color: #ffffff; padding: 15px; border-radius: 8px; margin: 10px 0; border: 1px solid #303f9f;">
        <h2 style="color: #64b5f6;">🎯 Memulai scraping untuk {len(target_queries)} kategori makanan</h2>
        <p style="color: #e3f2fd;"><strong style="color: #ffab40;">Total target gambar:</strong> {len(target_queries) * images_per_dish} gambar</p>
    </div>
    """))
    
    # Process each dish
    start_time = time.time()
    for i, (dish, query_list) in enumerate(target_queries.items(), 1):
        display(HTML(f"""
        <div style="background-color: #1a1a1a; padding: 10px; border-left: 4px solid #007bff; margin: 10px 0;">
            <h3>📍 Kategori {i}/{len(target_queries)}: {dish.upper()}</h3>
        </div>
        """))
        
        scrape_images_for_dish(dish, query_list, images_per_dish, save_directory, api_key)
        
        # Add delay between dishes
        if i < len(target_queries):
            print("⏱️ Menunggu 2 detik sebelum kategori berikutnya...")
            time.sleep(2)
    
    # Final summary
    elapsed_time = time.time() - start_time
    display(HTML(f"""
    <div style="background-color: #004d40; color: #ffffff; padding: 20px; border-radius: 10px; margin: 20px 0; text-align: center; border: 2px solid #00695c;">
        <h2 style="color: #4db6ac;">🎉 SEMUA TUGAS SCRAPING SELESAI! 🎉</h2>
        <p style="color: #b2dfdb;"><strong style="color: #26a69a;">Waktu total:</strong> {elapsed_time/60:.1f} menit</p>
        <p style="color: #b2dfdb;"><strong style="color: #26a69a;">Direktori output:</strong> <span style="color: #80cbc4;">{save_directory}</span></p>
    </div>
    """))


In [ ]:
run_scraper('config.yaml')